# N034 · 差分与批量区间更新

**目标：** 把重复区间更新转化为边界事件。

**先修：** N031, N033。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 一维/二维差分；前缀还原；端点抵消；闭区间转半开。

## 从问题到算法

一次区间加法对相邻差分只影响两处：在开始处启动增量，在结束后一位取消增量。扫描差分的前缀就重建每个位置的总增量。为避免端点歧义，本章更新接口统一采用零基半开 [l,r)；航班题的一基闭区间 [first,last] 转成 [first−1,last)。

## 最小实现

**本章接口：** `apply_range_add(n, updates)`、`car_pooling(trips, capacity)`

In [1]:
def apply_range_add(n,updates):
    diff=[0]*(n+1)
    for left,right,value in updates:
        if not 0<=left<=right<=n: raise IndexError('invalid range')
        diff[left]+=value; diff[right]-=value
    answer=[]; total=0
    for i in range(n):
        total+=diff[i]; answer.append(total)
    return answer

def car_pooling(trips,capacity):
    events={}
    for passengers,start,end in trips:
        events[start]=events.get(start,0)+passengers
        events[end]=events.get(end,0)-passengers
    load=0
    for position in sorted(events):
        load+=events[position]
        if load>capacity: return False
    return True

## 正确性、前提与复杂度

每个更新的前缀累积在l前为0、[l,r)为value、r后归零；线性叠加得到全部更新。拼车同站下车与上车可以同时汇总，区间按半开占用处理。

**代价：** 差分 O(n+q) 时间、O(n) 空间；稀疏拼车事件排序 O(q log q)、O(q) 空间。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

updates=[(0,2,10),(1,3,20),(1,5,25)]; diff=[0]*6
for l,r,v in updates: diff[l]+=v; diff[r]-=v
out=apply_range_add(5,updates)
show_table(['位置','差分事件','重建值'],[(i,diff[i],out[i] if i<5 else '边界哨兵') for i in range(6)])

位置,差分事件,重建值
0,10,10
1,45,55
2,-10,45
3,-20,25
4,0,25
5,-25,边界哨兵


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
bookings=[[1,2,10],[2,3,20],[2,5,25]]
assert apply_range_add(5,[(a-1,b,v) for a,b,v in bookings])==[10,55,45,25,25]
assert car_pooling([[2,1,5],[3,3,7]],4) is False
assert car_pooling([[2,1,5],[3,5,7]],3) is True
from random import Random
rng=Random(34)
for n in range(1,10):
    updates=[]; ref=[0]*n
    for _ in range(30):
        l=rng.randrange(n+1); r=rng.randrange(l,n+1); v=rng.randrange(-5,6)
        updates.append((l,r,v))
        for i in range(l,r): ref[i]+=v
    assert apply_range_add(n,updates)==ref
print('N034: 所有本章断言通过')

N034: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 实现矩形加值的二维差分。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 解释末端哨兵单元。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1109. Corporate Flight Bookings（canonical）
- 1094. Car Pooling（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。